# Databricks/PySpark for Data Analysts/Engineers — Module 6: Reading and Writing Data

So far our dataset has been created in memory (from `pandas` via `createDataFrame`). In
practice, data almost always starts and ends its journey in FILES -- this module shows
how Spark writes and reads them: CSV, JSON, and Parquet, write modes, and partitioning.

## Table of contents
1. [Writing to files: CSV, JSON, Parquet](#sec1)
2. [Reading: explicit vs. inferred schema](#sec2)
3. [Write modes](#sec3)
4. [Partitioning on write](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Writing to files: CSV, JSON, Parquet

`df.write.csv(...)`, `df.write.json(...)`, `df.write.parquet(...)` write a DataFrame to
files. IMPORTANT: Spark doesn't write a SINGLE file, but a WHOLE FOLDER, containing one
file per partition (plus a `_SUCCESS` file) -- a natural consequence of distributed
processing: every executor writes its own chunk separately.

In [ ]:
import os

base_path = "output_data"
os.makedirs(base_path, exist_ok=True)

products.write.mode("overwrite").csv(f"{base_path}/products_csv", header=True)
print(os.listdir(f"{base_path}/products_csv"))


In [ ]:
products.write.mode("overwrite").json(f"{base_path}/products_json")
products.write.mode("overwrite").parquet(f"{base_path}/products_parquet")
print("JSON and Parquet written.")


> ⚡ **Why Parquet is the default choice in Spark**
>
> **Parquet** is a columnar format (not row-based like CSV) with a built-in schema and compression -- Spark only has to read the columns actually needed by a query (*predicate/column pruning*), and doesn't have to guess types, since they're stored alongside the data. In professional practice, Parquet (and in Databricks -- Delta Lake, built directly on top of Parquet -- see Module 10) is the default choice for most pipelines, with CSV/JSON mainly used as an exchange format with external systems.

<a id="sec2"></a>
## 2. Reading: explicit vs. inferred schema

When reading CSV/JSON, Spark by default does NOT know the column types -- you either
have to explicitly ask it to guess them (`inferSchema=True`, expensive -- a double read
of the data, as mentioned in Module 2) or supply an EXPLICIT schema upfront (faster,
predictable).

In [ ]:
# Schema inference -- convenient, but expensive on large data
products_inferred = spark.read.csv(
    f"{base_path}/products_csv", header=True, inferSchema=True
)
products_inferred.printSchema()


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
])

products_explicit_schema = spark.read.csv(
    f"{base_path}/products_csv", header=True, schema=products_schema
)
products_explicit_schema.printSchema()


> ⚠️ **Parquet does NOT have this problem**
>
> Parquet writes the schema TOGETHER with the data -- when reading with `spark.read.parquet(...)`, Spark always knows the exact column types with no guessing or explicit schema needed. That's another reason (besides performance) Parquet is more convenient than CSV in most data pipelines.

In [ ]:
products_from_parquet = spark.read.parquet(f"{base_path}/products_parquet")
products_from_parquet.printSchema()


<a id="sec3"></a>
## 3. Write modes

The `.mode(...)` parameter tells Spark what to do if the target folder ALREADY EXISTS:

- `"overwrite"` -- deletes existing data and writes fresh (used above),
- `"append"` -- adds new files alongside existing ones,
- `"error"` (the default, when nothing is specified) -- raises an exception if the folder already exists,
- `"ignore"` -- if the folder exists, does nothing at all (silently skips the write).

In [ ]:
# append -- adding another batch of data to the same folder
first_half = customers.filter(customers.customer_id <= 150)
second_half = customers.filter(customers.customer_id > 150)

first_half.write.mode("overwrite").parquet(f"{base_path}/customers_parquet")
second_half.write.mode("append").parquet(f"{base_path}/customers_parquet")

loaded = spark.read.parquet(f"{base_path}/customers_parquet")
print(f"Row count after append: {loaded.count()}")


> ⚠️ **error and ignore are easy to confuse with no error at all**
>
> If you forget `.mode(...)` and the folder already exists, you'll get an exception (`AnalysisException: path already exists`) -- this is a safe default that protects against accidentally overwriting data. `"ignore"` is used less often and is easy to forget about -- make sure you really want to SILENTLY skip the write before using it.

<a id="sec4"></a>
## 4. Partitioning on write

`partitionBy("column")` splits written data into SUBFOLDERS by the column's value (e.g.
`category=Electronics/`, `category=Clothing/`, ...). Thanks to this, a later read with a
FILTER on that column can skip whole subfolders instead of reading everything
(*partition pruning*) -- very important for performance on large datasets.

In [ ]:
products.write.mode("overwrite").partitionBy("category").parquet(
    f"{base_path}/products_partitioned"
)
for folder in sorted(os.listdir(f"{base_path}/products_partitioned")):
    print(folder)


In [ ]:
# Reading with a filter on the partitioning column -- Spark will skip unneeded subfolders
electronics_products = spark.read.parquet(
    f"{base_path}/products_partitioned"
).filter("category = 'Electronics'")
electronics_products.explain()


> 🧱 **Choosing a partitioning column -- think it through**
>
> A good partitioning column has FEW unique values that are frequently used in WHERE filters (e.g. `category`, `region`, or a date truncated to day/month). Partitioning by a column with thousands of unique values (e.g. `customer_id`) creates thousands of tiny subfolders -- this slows things down instead of speeding them up. We come back to this in more detail in Module 9 (optimization).

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- writing to CSV/JSON/Parquet (`df.write.csv/json/parquet(...)`) -- always as a folder, never a single file,
- reading with an explicit vs. inferred schema (`inferSchema`) -- and why Parquet sidesteps this problem,
- write modes: `overwrite`, `append`, `error`, `ignore`,
- partitioning on write (`partitionBy`) and *partition pruning* on read.

### 📝 Exercise 1: Writing and reading orders

Write `orders` to Parquet (`overwrite`), then read it back and print `printSchema()` and `count()`.

<details>
<summary>Show solution</summary>

```python
orders.write.mode("overwrite").parquet("output_data/orders_parquet")
loaded = spark.read.parquet("output_data/orders_parquet")
loaded.printSchema()
print(f"Row count: {loaded.count()}")
```

</details>

### 📝 Exercise 2: An explicit schema for employees

Write `employees` to CSV with a header, then read the file back with an EXPLICIT schema (`StructType`) instead of `inferSchema`.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

employees.write.mode("overwrite").csv("output_data/employees_csv", header=True)

schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("region", StringType(), True),
    StructField("department", StringType(), True),
])
spark.read.csv("output_data/employees_csv", header=True, schema=schema).printSchema()
```

</details>

### 📝 Exercise 3: Appending two batches

Split `products` into two parts (`product_id <= 20` and `product_id > 20`), write the first with `overwrite`, the second with `append` to the same folder, and check the final row count.

<details>
<summary>Show solution</summary>

```python
part1 = products.filter(products.product_id <= 20)
part2 = products.filter(products.product_id > 20)

part1.write.mode("overwrite").parquet("output_data/products_append_test")
part2.write.mode("append").parquet("output_data/products_append_test")

result = spark.read.parquet("output_data/products_append_test")
print(f"Row count: {result.count()}")
```

</details>

### 📝 Exercise 4: Partitioning orders by status

Write `orders` with `partitionBy("status")` and print the names of the resulting subfolders.

<details>
<summary>Show solution</summary>

```python
import os

orders.write.mode("overwrite").partitionBy("status").parquet(
    "output_data/orders_partitioned"
)
for folder in sorted(os.listdir("output_data/orders_partitioned")):
    print(folder)
```

</details>

> 🔥 **Challenge: a full file -> transformation -> partitioned file pipeline**
>
> Read `products` from the Parquet written in section 1, add a `price_tier` column (as in Module 3, `when/otherwise`: `'Cheap'`/`'Mid'`/`'Expensive'`), write the result partitioned by BOTH `category` AND `price_tier` at once (`partitionBy("category", "price_tier")`), and finally read it back and count products in each combination.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql import functions as F

loaded_products = spark.read.parquet("output_data/products_parquet")

products_with_tier = loaded_products.withColumn(
    "price_tier",
    F.when(F.col("price") < 100, "Cheap")
    .when(F.col("price") <= 500, "Mid")
    .otherwise("Expensive"),
)

products_with_tier.write.mode("overwrite").partitionBy(
    "category", "price_tier"
).parquet("output_data/products_full_pipeline")

result = spark.read.parquet("output_data/products_full_pipeline")
result.groupBy("category", "price_tier").count().orderBy(
    "category", "price_tier"
).show(20)
```

Partitioning by MULTIPLE columns creates nested subfolders: category=X/price_tier=Y/ -- every unique combination gets its own path.

</details>

## What's next?

In **Module 7** we'll move to window functions (`Window`): `row_number`, `rank`, running
totals, and `lag`/`lead` -- the equivalent of the window functions from Module 11 of the
SQL course, now in PySpark.